# 04 — Routing and admission: who decides, and what it costs

Three decisions happen before a token is generated, and this notebook makes
each one visible:

1. **guard** — is this model served, is `max_tokens` sane
2. **admit** — is there KV headroom, or is this a 503
3. **place** — which replica, and can prefix reuse make it free

## What this fleet does and does not do

It runs `LAB_TOPOLOGY=aggregated`: one full engine per card, each serving
both phases, with the router choosing between them. **There is no KV hop.**
`queue.py` short-circuits when the prefill and decode pools are the same
object, so the ledger on `:50051` stays at zero by design.

That is deliberate rather than missing. A real prefill/decode split needs
placement, orchestration **and transfer**, and transfer lives inside vLLM
behind `--kv-transfer-config` — which names an RDMA device and a cluster DNS
name, so it cannot be passed on a rented box. Without it decode re-prefills
the whole prompt and one answer costs two prefills, which is strictly worse
than not splitting. See [`docs/project/topology.md`](../../docs/project/topology.md).

So what is measured here is real: placement across replicas, prefix reuse,
KV pressure, and admission shedding.

## Before you run it

```bash
make fleet-runpod              # the fleet, on the box
make fleet-runpod-ssh          # the tunnel, left open in its own terminal
```

Load is driven from here through the tunnel, which is fine for *counting
decisions* — every number below is a counter or a gauge read from the
cluster, not a latency. For latency, run `make load`, which drives the box
locally so your WAN round-trip is not inside every TTFT.

In [ ]:
import concurrent.futures as cf
import json, sys, time, urllib.error, urllib.parse, urllib.request
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == '04-routing-admission' else Path.cwd()
sys.path.insert(0, str(ROOT / 'benchmarking'))
%load_ext autoreload
%autoreload 2

from prom import Prom
from utils import Report, filler, percentile

GATEWAY = 'http://127.0.0.1:8080'

# Prometheus is on :9090 on the process path and :31490 behind a k8s
# NodePort. Try both rather than make you edit a cell — a notebook that
# reports 'cluster down' because of a port number wastes the run.
def find_prom():
    for base in ('http://127.0.0.1:9090', 'http://127.0.0.1:31490'):
        p = Prom(base=base)
        if p.up():
            return p, base
    return None, None

prom, prom_base = find_prom()
report = Report('04-routing-admission')
TRACES = ROOT / 'benchmarking' / '04-routing-admission' / 'traces'
TRACES.mkdir(exist_ok=True)

def v(expr):
    """One number from an instant query, 0.0 when absent."""
    rows = prom.query(expr) if prom else []
    return float(rows[0][1]) if rows else 0.0

def series(expr, label):
    """{label_value: number} — for the per-replica tables below."""
    return {m.get(label, '?'): val for m, val in (prom.query(expr) if prom else [])}

def get_json(url, timeout=10):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return json.load(r)
    except Exception:
        return None

print(f'prometheus : {prom_base or "NOT REACHABLE — is the tunnel open?"}')
models = get_json(f'{GATEWAY}/v1/models')
served = [m['id'] for m in (models or {}).get('data', [])]
print(f'gateway    : {", ".join(served) if served else "NOT REACHABLE"}')

## 1. The fleet — who can take work

`place()` is only a decision if more than one replica can serve. With one
replica it degenerates to routing, and every panel downstream becomes a
description of a single process.

In [ ]:
# Replica identity comes from the gateway's own view, not from guessing
# ports: `orch_replica_*` carries the pod name pools.py assigned, which is
# what every later table joins on.
healthy = series('max by (pod) (orch_replica_healthy)', 'pod')
pools   = {m.get('pod'): m.get('pool') for m, _ in
           (prom.query('orch_replica_kv_free_ratio') if prom else [])}
kv_free = series('min by (pod) (orch_replica_kv_free_ratio)', 'pod')

rows = [[pod, pools.get(pod, '?'), 'yes' if healthy.get(pod, 0) >= 1 else 'NO',
         f'{kv_free.get(pod, 0):.3f}'] for pod in sorted(healthy)]
report.heading('1. The fleet')
report.table(['replica', 'pool', 'healthy', 'KV free'], rows,
             note='pool is the same name on both sides when aggregated — one pool, both phases')

n_replicas = len(rows)
if n_replicas >= 2:
    report.text(f'{n_replicas} replicas — place() is choosing, not routing.')
else:
    report.text(f'{n_replicas} replica: place() has nothing to choose between. '
                'Everything below still runs, but routing is not being demonstrated.')

## 2. Is the threshold reachable? — the INERT check

**Run this before any load test.** The admission policy fires when
`vllm:kv_cache_usage_perc` crosses its threshold (0.90 by default). If the KV
pool is large enough that the pool can never fill, the policy never fires,
and the run comes back with numbers indistinguishable from having no policy
at all. That outcome is **INERT** — not a failure of the policy, a failure of
the experiment design.

The arithmetic: a request holds roughly its whole prompt plus what it has
generated. So the pool fills when

```
concurrent_requests x tokens_per_request  >=  threshold x pool_tokens
```

and `MAX_NUM_SEQS` caps `concurrent_requests` per replica. If that product
cannot reach the threshold, lower `SLICE_MIB` — it shrinks the pool, which is
the only lever that does not require more load than you can generate.

In [ ]:
# Read the pool size from the engine itself rather than deriving it from a
# model card: vLLM reports the blocks it actually allocated, after weights
# and activations took their share.
THRESHOLD = 0.90          # ADMISSION_THRESHOLD
KIB_PER_TOKEN = 256      # Qwen2.5-32B: 64 layers x 8 kv-heads x 128 dim x 2 (k+v) x 2 bytes

def engine_ports(n):
    return [8100 + i for i in range(max(n, 1))]

pool_rows, pool_tokens = [], []
for port in engine_ports(n_replicas):
    txt = None
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{port}/metrics', timeout=10) as r:
            txt = r.read().decode()
    except Exception:
        pass
    if not txt:
        pool_rows.append([f':{port}', '—', '—', 'not reachable'])
        continue
    # cache_config_info carries num_gpu_blocks and block_size as labels.
    info = [l for l in txt.splitlines() if l.startswith('vllm:cache_config_info')]
    def label(key):
        for l in info:
            if f'{key}="' in l:
                return l.split(f'{key}="')[1].split('"')[0]
        return None
    blocks, bsize = label('num_gpu_blocks'), label('block_size')
    if blocks and bsize:
        toks = int(float(blocks)) * int(float(bsize))
        pool_tokens.append(toks)
        pool_rows.append([f':{port}', f'{int(float(blocks)):,}', f'{bsize}', f'{toks:,} tokens'])
    else:
        pool_rows.append([f':{port}', '—', '—', 'cache_config_info absent'])

report.heading('2. Is the threshold reachable?')
report.table(['engine', 'gpu blocks', 'block size', 'KV pool'], pool_rows,
             note='read from vllm:cache_config_info — what was allocated, not what was asked for')

if pool_tokens:
    per_replica = min(pool_tokens)
    need = THRESHOLD * per_replica
    gib = per_replica * KIB_PER_TOKEN / (1024 * 1024)
    report.text(f'Smallest pool: {per_replica:,} tokens (~{gib:.1f} GiB at {KIB_PER_TOKEN} KiB/token).')
    report.text(f'To reach {THRESHOLD:.2f} one replica must hold {need:,.0f} tokens at once.')
    rows = [[f'{t:,}', f'{need / t:,.0f}'] for t in (500, 1000, 2000, 4000, 8000)]
    report.table(['tokens per request', 'concurrent requests needed'], rows,
                 note='against MAX_NUM_SEQS (64 by default) — above that, the pool cannot fill')
else:
    report.text('No engine reported cache_config_info; skipping the arithmetic.')

## 3. One request, and the decisions it passed through

Counter deltas around a single completion. This is the clearest view of the
control plane: one request, and exactly which decisions were recorded.

In [ ]:
DECISIONS = {
    'orch_requests_total':  'arrived (passed the guard)',
    'orch_place_total':     'placed on a pool',
    'orch_pick_total':      'a replica was picked',
    'orch_sticky_total':     'picked for prefix reuse',
    'orch_completed_total': 'completed',
    'orch_shed_total':      'shed for capacity (503)',
    'orch_kv_transfer_total': 'KV hop recorded (a decision, not bytes)',
}

def snapshot():
    return {k: v(f'sum({k})') for k in DECISIONS}

def ask(prompt, max_tokens=40, model='text', timeout=240, tools=None):
    body = {'model': model, 'max_tokens': max_tokens, 'temperature': 0.0,
            'messages': [{'role': 'user', 'content': prompt}]}
    if tools:
        body['tools'] = tools
    req = urllib.request.Request(f'{GATEWAY}/v1/chat/completions',
                                 json.dumps(body).encode(),
                                 {'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, json.load(r)
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read() or b'{}')
    except Exception as e:
        return 0, {'error': str(e)}

before = snapshot()
status, body = ask('In one short sentence, what is prefill in LLM inference?')
time.sleep(7)        # one scrape interval, so the counters have moved
after = snapshot()

rows = [[DECISIONS[k], f'{after[k] - before[k]:+.0f}'] for k in DECISIONS
        if after[k] != before[k]]
report.heading('3. One request')
report.text(f'HTTP {status}.')
report.table(['decision', 'delta'], rows or [['nothing moved', '0']],
             note='a decision with no delta is simply one this request did not need')

## 4. place() spreading work across replicas

Concurrent requests, then per-replica counters. Two replicas both moving is
placement working; one flat at zero means it is not — usually because the
other replica is unhealthy or saturating and the router is avoiding it.

In [ ]:
PROMPT_TOKENS = 1200
CONCURRENCY = 12

def distinct(i):
    # Distinct prefixes on purpose: a shared prefix would make sticky routing
    # send everything to one replica, which is the NEXT section's point and
    # would hide spreading here.
    return f'[{i}] {filler(PROMPT_TOKENS)} Reply with one short sentence.'

before = snapshot()
with cf.ThreadPoolExecutor(CONCURRENCY) as ex:
    codes = Counter(s for s, _ in ex.map(lambda i: ask(distinct(i), max_tokens=60),
                                         range(CONCURRENCY)))
time.sleep(7)
after = snapshot()

running = series('sum by (pod) (vllm:num_requests_running)', 'pod')
succ    = series('sum by (pod) (vllm:request_success_total)', 'pod')
report.heading('4. Placement across replicas')
report.text(f'{CONCURRENCY} concurrent requests -> ' +
            ', '.join(f'{n}x HTTP {c}' for c, n in sorted(codes.items())))
report.table(['replica', 'succeeded (total)', 'running now'],
             [[pod, f'{succ.get(pod, 0):,.0f}', f'{running.get(pod, 0):.0f}']
              for pod in sorted(set(succ) | set(running))] or [['no series', '—', '—']],
             note='succeeded is cumulative; both replicas climbing is place() spreading load')
report.table(['decision', 'delta'],
             [[DECISIONS[k], f'{after[k] - before[k]:+.0f}'] for k in DECISIONS
              if after[k] != before[k]] or [['nothing moved', '0']])

## 5. Prefix reuse — the cheapest outcome available

A request sent back to the replica that already holds its prompt skips the
prefill entirely. No hop, no recompute. This is what the KV index in
`router/kvbus.py` is genuinely for, and on this topology it is the only KV
saving on offer — so it is worth measuring rather than asserting.

The same long prefix, several times. `orch_sticky_total` should move, and
`vllm:prompt_tokens_cached_total` should grow much faster than
`vllm:prompt_tokens_total`.

In [ ]:
SHARED = filler(1500)

def cached_vs_total():
    return (v('sum(vllm:prompt_tokens_cached_total)'),
            v('sum(vllm:prompt_tokens_total)'))

sticky_before = v('sum(orch_sticky_total)')
cached_before, total_before = cached_vs_total()

# Sequential, not concurrent: the first has to finish and register its prefix
# before the rest can be routed back to it.
for i in range(6):
    ask(f'{SHARED}\n\nQuestion {i}: reply with one short sentence.', max_tokens=40)
time.sleep(7)

sticky_after = v('sum(orch_sticky_total)')
cached_after, total_after = cached_vs_total()
d_cached, d_total = cached_after - cached_before, total_after - total_before
hit = v('sum(rate(vllm:prefix_cache_hits_total[5m])) / '
        'clamp_min(sum(rate(vllm:prefix_cache_queries_total[5m])), 1)')

report.heading('5. Prefix reuse')
report.table(['measure', 'value'], [
    ['sticky picks', f'{sticky_after - sticky_before:+.0f}'],
    ['prompt tokens sent', f'{d_total:,.0f}'],
    ['of those, already cached', f'{d_cached:,.0f}'],
    ['prefill avoided', f'{(d_cached / d_total * 100) if d_total else 0:.0f}%'],
    ['prefix cache hit rate (5m)', f'{hit:.3f}'],
], note='6 requests sharing a ~1500-token prefix')
if d_total and d_cached / d_total > 0.5:
    report.text('Most of the prompt was never prefilled again — reuse is working.')
else:
    report.text('Little reuse. Either the prefix is shorter than a cache block, '
                'or requests were spread across replicas faster than the prefix registered.')

## 6. Load, and the signal admission reads

Now push concurrency up and sample `vllm:kv_cache_usage_perc` while it runs.
This is the only number the KV policy looks at, so this plot is the
experiment: if the line never approaches the threshold, nothing the policy
does afterwards means anything.

Driving this from the laptop limits how much concurrency you can actually
sustain. If KV stays flat and low, use `make load` (which runs on the box)
before concluding the policy is inert.

In [ ]:
RAMP = [8, 16, 24, 32]
SAMPLE_EVERY = 2.0

samples, stop = [], False

def watch():
    while not stop:
        samples.append((time.time(),
                        v('max(vllm:kv_cache_usage_perc)'),
                        v('sum(vllm:num_requests_running)'),
                        v('sum(vllm:num_requests_waiting)')))
        time.sleep(SAMPLE_EVERY)

import threading
watcher = threading.Thread(target=watch, daemon=True)
watcher.start()

before = snapshot()
stage_rows = []
for users in RAMP:
    t0 = time.time()
    with cf.ThreadPoolExecutor(users) as ex:
        codes = Counter(s for s, _ in ex.map(
            lambda i: ask(f'[{users}-{i}] {filler(2000)} One short sentence please.',
                          max_tokens=120), range(users)))
    peak = max((s[1] for s in samples), default=0.0)
    stage_rows.append([users, f'{time.time() - t0:.0f}s',
                       ', '.join(f'{n}x{c}' for c, n in sorted(codes.items())),
                       f'{peak:.3f}'])
    print(f'  {users:>3} users  peak KV {peak:.3f}  {dict(codes)}')

stop = True
time.sleep(SAMPLE_EVERY + 0.5)
after = snapshot()

report.heading('6. Load and KV pressure')
report.table(['concurrent', 'wall', 'responses', 'peak KV (cumulative)'], stage_rows)
(TRACES / 'kv-samples.json').write_text(json.dumps(
    [{'t': t, 'kv': k, 'running': r, 'waiting': w} for t, k, r, w in samples], indent=2))
print(f'  {len(samples)} samples -> {TRACES / "kv-samples.json"}')

In [ ]:
# The plot, because a peak number hides the shape. A line that rises and
# plateaus below the threshold is the INERT signature.
import matplotlib.pyplot as plt

if samples:
    t0 = samples[0][0]
    xs = [s[0] - t0 for s in samples]
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.plot(xs, [s[1] for s in samples], label='KV usage (fullest replica)')
    ax.axhline(THRESHOLD, ls='--', lw=1, color='crimson',
               label=f'admission threshold {THRESHOLD}')
    ax.set_xlabel('seconds'); ax.set_ylabel('KV used'); ax.set_ylim(0, 1)
    ax2 = ax.twinx()
    ax2.plot(xs, [s[2] for s in samples], lw=1, alpha=0.6, color='grey', label='running')
    ax2.plot(xs, [s[3] for s in samples], lw=1, alpha=0.6, ls=':', color='grey', label='waiting')
    ax2.set_ylabel('requests')
    ax.legend(loc='upper left', fontsize=8); ax2.legend(loc='upper right', fontsize=8)
    ax.set_title('Did the signal the policy reads ever reach the threshold?')
    fig.tight_layout()
    fig.savefig(TRACES / 'kv-pressure.png', dpi=120)
    plt.show()
else:
    print('  no samples — was Prometheus reachable?')

## 7. Admission — the decision to refuse

A 503 from `orch_shed_total` is the policy working. A 429 is **not**: that is
the per-tenant quota, which is client-attributable and says nothing about
capacity. The two are easy to confuse in a report, and only one of them is
evidence about the cluster.

In [ ]:
gate = {m.get('policy', '?'): val for m, val in
        (prom.query('max by (policy) (admitperf_gate_active)') if prom else [])}
shed_by_reason = series('sum by (reason) (orch_shed_total)', 'reason')

report.heading('7. Admission')
report.table(['policy', 'active'], [[k, f'{x:.0f}'] for k, x in gate.items()] or
             [['none reported', '0']],
             note='policy=none means ADMISSION_POLICY is unset and nothing is being shed')
report.table(['shed reason', 'total'],
             [[k, f'{x:,.0f}'] for k, x in sorted(shed_by_reason.items())] or
             [['nothing shed', '0']])

peak_kv = max((s[1] for s in samples), default=0.0)
shed_delta = after['orch_shed_total'] - before['orch_shed_total']

if 'none' in gate and gate.get('none', 0) >= 1:
    verdict = ('NO POLICY — ADMISSION_POLICY is unset, so the gate is off. Set it in '
               '.env and re-run `make fleet-runpod`; the gateway reads it at startup.')
elif shed_delta > 0:
    verdict = (f'POLICY FIRED — {shed_delta:.0f} request(s) shed with peak KV {peak_kv:.3f}. '
               'This run is evidence.')
elif peak_kv >= THRESHOLD:
    verdict = (f'Peak KV {peak_kv:.3f} reached the threshold but nothing was shed. '
               'Check ADMISSION_ENFORCE and the policy name.')
else:
    verdict = (f'INERT — peak KV was {peak_kv:.3f}, never near {THRESHOLD}. The policy was '
               'never asked a hard question, so these numbers cannot distinguish it from '
               'no policy at all. Lower SLICE_MIB, or drive load from the box with `make load`.')
report.text(verdict)
print('\n  ' + verdict)

## 8. What this run proved

And, as importantly, what it did not. The honest list is short, which is the
point of writing it down.

In [ ]:
report.heading('8. Verdict')
report.table(['claim', 'shown by'], [
    ['placement is a real choice', f'{n_replicas} healthy replicas, both serving (section 4)'],
    ['prefix reuse avoids prefill', 'cached vs total prompt tokens (section 5)'],
    ['KV pressure is observable', f'peak {max((s[1] for s in samples), default=0.0):.3f} (section 6)'],
    ['admission decides', 'shed reasons and the gate series (section 7)'],
], note='every row is a counter or gauge read from the cluster, not an assertion')
report.table(['NOT shown', 'why'], [
    ['a KV transfer', 'no --kv-transfer-config: one pool, so no hop is attempted at all'],
    ['prefill/decode isolation', 'every engine serves both phases here, by design'],
    ['end-to-end latency', 'driven through the tunnel — use `make load` on the box'],
], note='docs/project/topology.md has the full argument')

path = report.save()
print(f'\n  report -> {path}')